In [1]:
# ============================================
# DEEPFAKE DETECTION - 3-DATASET T=32 TRAINING
# Experiment A3: Temporal Only
# ============================================
# Datasets: DFDC02 + DFD01 + CelebDF (all preprocessed with T=32)
# GPU: P100 recommended (16GB)
#
# Required Kaggle inputs:
#   - dfdc02-preprocessed-t32
#   - dfd01-preprocessed-t32
#   - celebdf-preprocessed-t32
#   - 3ds-t32-a1-results (output from A1 notebook, contains split file)

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write training script
train_script = r'''
import os, sys, time, json, gc, shutil
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f'numpy version: {np.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), 'No GPU detected!'
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# =============================================
# Restore split file from A1 output
# =============================================
SPLIT_FILENAME = 'split_seed42_3ds_t32.json'

def restore_split_from_a1():
    """Search /kaggle/input for the split file from A1 output."""
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == SPLIT_FILENAME:
                src = os.path.join(root, f)
                os.makedirs('./splits', exist_ok=True)
                dst = os.path.join('./splits', SPLIT_FILENAME)
                shutil.copy2(src, dst)
                print(f'[SPLIT] Restored split file from {src}')
                return True
    return False

split_found = restore_split_from_a1()
if not split_found:
    print('[SPLIT] WARNING: Split file from A1 not found! Will create new split.')

# =============================================
# Find ALL T=32 datasets in /kaggle/input
# =============================================
TARGET_T = 32
print(f'\n=== Searching for T={TARGET_T} datasets ===')
found_datasets = []
for root, dirs, files in os.walk('/kaggle/input'):
    dirname_lower = os.path.basename(root).lower()
    if '_16' in dirname_lower or '-16' in dirname_lower:
        continue
    # Skip result datasets
    if '3ds-t32-a' in dirname_lower.replace('_', '-'):
        continue

    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            sample_dir = None
            for label in ('real', 'fake'):
                lp = os.path.join(root, label)
                for vd in os.listdir(lp):
                    vdp = os.path.join(lp, vd)
                    if os.path.isdir(vdp):
                        sample_dir = vdp
                        break
                if sample_dir:
                    break
            if sample_dir:
                n_frames = len([f for f in os.listdir(sample_dir) if f.endswith('.jpg')])
                if n_frames > 0 and abs(n_frames - TARGET_T) > TARGET_T * 0.5:
                    print(f'  SKIP: {root} (frames={n_frames}, expected ~{TARGET_T})')
                    continue

            path_lower = root.lower()
            if 'dfd01' in path_lower or 'dfd-01' in path_lower or 'dfd_01' in path_lower:
                ds_name = 'dfd01'
            elif 'dfdc02' in path_lower or 'dfdc-02' in path_lower or 'dfdc' in path_lower:
                ds_name = 'dfdc02'
            elif 'celeb' in path_lower:
                ds_name = 'celebdf'
            else:
                ds_name = os.path.basename(root).replace('-', '_')

            if any(d['name'] == ds_name for d in found_datasets):
                print(f'  SKIP duplicate: {ds_name} at {root}')
                continue

            found_datasets.append({'path': root, 'name': ds_name, 'real': rc, 'fake': fc})
            print(f'  Found: {ds_name} at {root} (real={rc}, fake={fc})')

if len(found_datasets) < 3:
    print(f'WARNING: Expected 3 datasets, found {len(found_datasets)}!')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    if len(found_datasets) == 0:
        sys.exit(1)

dataset_roots = '+'.join(d['path'] for d in found_datasets)
dataset_names = '+'.join(d['name'] for d in found_datasets)
total_videos = sum(d['real'] + d['fake'] for d in found_datasets)
print(f'\nCombined: {dataset_names}')
print(f'Total videos: {total_videos}')

# =============================================
# Run experiment A3: Temporal Only
# =============================================
from config import Config
from train import train
from utils import save_metrics

EXP_NAME = 'A3_temporal_only'

print(f'\n{"="*70}')
print(f'[1/1] {EXP_NAME} (model_type=temporal, T=32, 3-dataset)')
print(f'{"="*70}\n')

cfg = Config()
cfg.dataset_root = dataset_roots
cfg.dataset_name = dataset_names
cfg.model_type = 'temporal'
cfg.fusion_type = 'adaptive'
cfg.num_frames = 32
cfg.min_frames_per_video = 24
cfg.max_epochs = 30
cfg.batch_size = 8
cfg.patience = 7
cfg.device = 'auto'
cfg.use_amp = True
cfg.num_workers = 2
cfg.pin_memory = True
cfg.splits_dir = './splits'
cfg.output_dir = './experiments'
cfg.seed = 42
cfg.unfreeze_last_n_blocks = 2
cfg.split_filename = SPLIT_FILENAME

split_exists = os.path.exists(os.path.join('./splits', SPLIT_FILENAME))
if split_exists:
    cfg.split_mode = 'fixed'
    cfg.save_split = False
    print(f'Using existing split file: {SPLIT_FILENAME}')
else:
    cfg.split_mode = 'random'
    cfg.save_split = True
    print(f'WARNING: Creating new split (A1 split not found)')

t0 = time.time()
all_results = []
try:
    metrics = train(cfg)
    metrics['experiment_name'] = EXP_NAME
    metrics['status'] = 'success'
    metrics['duration_min'] = round((time.time() - t0) / 60, 1)
    all_results.append(metrics)
    test = metrics.get('test', {})
    print(f'\n[OK] {EXP_NAME} done in {metrics["duration_min"]} min')
    print(f'     AUC={test.get("auc",0):.4f}  Acc={test.get("accuracy",0):.4f}  F1={test.get("f1",0):.4f}')
except Exception as e:
    elapsed = round((time.time() - t0) / 60, 1)
    print(f'\n[FAIL] {EXP_NAME} after {elapsed} min: {e}')
    import traceback; traceback.print_exc()
    all_results.append({
        'experiment_name': EXP_NAME,
        'status': 'failed',
        'error': str(e),
        'duration_min': elapsed,
    })

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

save_metrics(all_results, './experiments/all_results_3ds_t32_a3.json')

# =============================================
# Summary
# =============================================
print(f'\n{"="*70}')
print('RESULTS SUMMARY - A3 Temporal Only (DFDC02+DFD01+CelebDF, T=32)')
print(f'{"="*70}')
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f'{r["experiment_name"]:<25} AUC={t.get("auc",0):.4f}  Acc={t.get("accuracy",0):.4f}  '
              f'F1={t.get("f1",0):.4f}  EER={t.get("eer",0):.4f}  Epoch={be}  Time={r.get("duration_min",0):.1f}m')
    else:
        print(f'{r["experiment_name"]:<25} FAILED: {r.get("error","")[:60]}')

os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('cp -r splits/ /kaggle/working/splits/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_3ds_t32_a3.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f'\nresults_3ds_t32_a3.tar.gz ready for download')
'''

with open('/kaggle/working/run_training_3ds_t32_a3.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training
print('\n=== Starting A3 Temporal Only Training (T=32, 3-dataset) ===')
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_training_3ds_t32_a3.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'\nTraining subprocess exited with code: {result.returncode}')

if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print('Results copied to /kaggle/working/experiments/')
if os.path.exists('/kaggle/working/results_3ds_t32_a3.tar.gz'):
    print('results_3ds_t32_a3.tar.gz is ready for download')

=== Installing dependencies ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 95.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

Dependencies installed.


...

Project cloned.

=== Starting A3 Temporal Only Training (T=32, 3-dataset) ===


2026-04-04 08:56:41 | INFO | ======================================================================
2026-04-04 08:56:41 | INFO | Эксперимент: celebdf+dfd01+dfdc02_temporal_seed42_bs8_T32
2026-04-04 08:56:41 | INFO | Dataset: celebdf+dfd01+dfdc02
2026-04-04 08:56:41 | INFO | Model: temporal
2026-04-04 08:56:41 | INFO | Fusion: adaptive
2026-04-04 08:56:41 | INFO | Seed: 42
2026-04-04 08:56:41 | INFO | Device: cuda
2026-04-04 08:56:41 | INFO | Batch size: 8
2026-04-04 08:56:41 | INFO | Epochs: 30
2026-04-04 08:56:41 | INFO | Primary metric: auc
2026-04-04 08:56:41 | INFO | ======================================================================
2026-04-04 08:56:41 | INFO | Загрузка данных...
2026-04-04 08:59:15 | INFO | Train: 6966 videos | Val: 1492 | Test: 1495
2026-04-04 08:59:15 | INFO | Создание модели...
2026-04-04 08:59:19 | INFO | Warmup: spatial backbone заморожен (эпохи 1..5)
2026-04-04 08:59:19 | INFO | Параметры: всего=6,220,669, обучаемых=6,220,669
2026-04-04 08:59:19 | INFO |

numpy version: 1.26.4
PyTorch: 2.2.2+cu121
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB
[SPLIT] Restored split file from /kaggle/input/notebooks/alexandertarakanov/kaggle-train-3ds-t32-a1/project/splits/split_seed42_3ds_t32.json

=== Searching for T=32 datasets ===
  Found: celebdf at /kaggle/input/datasets/alexandertarakanov/celebdf-preprocessed-t32 (real=589, fake=2710)
  Found: dfd01 at /kaggle/input/datasets/alexandertarakanov/dfd01-preprocessed-t32 (real=363, fake=3051)
  Found: dfdc02 at /kaggle/input/datasets/alexandertarakanov/dfdc02-preprocessed-t32/preprocessed_DFDC02_32 (real=1699, fake=1541)

Combined: celebdf+dfd01+dfdc02
Total videos: 9953

[1/1] A3_temporal_only (model_type=temporal, T=32, 3-dataset)

Using existing split file: split_seed42_3ds_t32.json
[ДАННЫЕ] Всего видео: 3299 (real: 589, fake: 2710)
[ДАННЫЕ] Всего видео: 3414 (real: 363, fake: 3051)
[ДАННЫЕ] Всего видео: 3240 (real: 1699, fake: 1541)
[MULTI-DATA] Всего видео: 9953 (real: 2651, fake: 7302)
  [celebdf

2026-04-04 09:01:30 | INFO |   [batch 87/870] avg_loss=0.6458
2026-04-04 09:03:26 | INFO |   [batch 174/870] avg_loss=0.6486
2026-04-04 09:05:24 | INFO |   [batch 261/870] avg_loss=0.6461
2026-04-04 09:07:31 | INFO |   [batch 348/870] avg_loss=0.6483
2026-04-04 09:09:37 | INFO |   [batch 435/870] avg_loss=0.6501
2026-04-04 09:12:02 | INFO |   [batch 522/870] avg_loss=0.6493
2026-04-04 09:14:43 | INFO |   [batch 609/870] avg_loss=0.6485
2026-04-04 09:17:05 | INFO |   [batch 696/870] avg_loss=0.6489
2026-04-04 09:19:22 | INFO |   [batch 783/870] avg_loss=0.6475
2026-04-04 09:21:30 | INFO |   [batch 870/870] avg_loss=0.6456
2026-04-04 09:26:36 | INFO | Epoch 01/30 | Train Loss: 0.6456 | Train AUC: 0.5204 | Val Loss: 0.6306 | Val AUC: 0.5745 | Val Acc: 0.6971 | Val F1: 0.8102 | LR backbone: 0.000020 | LR head: 0.000060 | Time: 1636.5s
2026-04-04 09:26:36 | INFO | Лучшая модель сохранена: auc = 0.5745
2026-04-04 09:28:18 | INFO |   [batch 87/870] avg_loss=0.5921
2026-04-04 09:29:49 | INFO |


[OK] A3_temporal_only done in 534.5 min
     AUC=0.9883  Acc=0.9538  F1=0.9687

RESULTS SUMMARY - A3 Temporal Only (DFDC02+DFD01+CelebDF, T=32)
A3_temporal_only          AUC=0.9883  Acc=0.9538  F1=0.9687  EER=0.0458  Epoch=24  Time=534.5m

results_3ds_t32_a3.tar.gz ready for download

Training subprocess exited with code: 0
Results copied to /kaggle/working/experiments/
results_3ds_t32_a3.tar.gz is ready for download
